In [6]:
# Instalando dependências
!pip install pandas scikit-learn joblib

In [7]:
# Imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

In [9]:
# Célula diagnóstico do data set
df_raw = pd.read_csv('Algerian_forest_fires_dataset.csv', header=None)
print(df_raw.head(20))
print("\n---")
print(f"Shape: {df_raw.shape}")
print("\nÚltimas linhas:")
print(df_raw.tail(10))

     0      1     2            3    4    5      6     7     8     9    10  \
0   day  month  year  Temperature   RH   Ws  Rain   FFMC   DMC    DC  ISI   
1     1      6  2012           29   57   18      0  65.7   3.4   7.6  1.3   
2     2      6  2012           29   61   13    1.3  64.4   4.1   7.6    1   
3     3      6  2012           26   82   22   13.1  47.1   2.5   7.1  0.3   
4     4      6  2012           25   89   13    2.5  28.6   1.3   6.9    0   
5     5      6  2012           27   77   16      0  64.8     3  14.2  1.2   
6     6      6  2012           31   67   14      0  82.6   5.8  22.2  3.1   
7     7      6  2012           33   54   13      0  88.2   9.9  30.5  6.4   
8     8      6  2012           30   73   15      0  86.6  12.1  38.3  5.6   
9     9      6  2012           25   88   13    0.2  52.9   7.9  38.8  0.4   
10   10      6  2012           28   79   12      0  73.2   9.5  46.3  1.3   
11   11      6  2012           31   65   14      0  84.5  12.5  54.3    4   

In [10]:
# Carrega dataset corrigido
df_raw = pd.read_csv('Algerian_forest_fires_dataset.csv', header=None)

# Primeira linha real tem os nomes das colunas
df_raw.columns = df_raw.iloc[0]
df = df_raw[1:].copy()
df.columns = df.columns.str.strip()
df = df.reset_index(drop=True)

# Remove linhas que sejam cabeçalho duplicado no meio do arquivo
df = df[df['Temperature'] != 'Temperature'].copy()
df = df[df['Classes'].notna()].copy()
df['Classes'] = df['Classes'].str.strip().str.lower()
df = df[df['Classes'].isin(['fire', 'not fire'])]

print(df['Classes'].value_counts())
print(df.shape)
print(df.head(3))

Classes
fire        137
not fire    106
Name: count, dtype: int64
(243, 14)
0 day month  year Temperature  RH  Ws  Rain  FFMC  DMC   DC  ISI  BUI  FWI  \
0   1     6  2012          29  57  18     0  65.7  3.4  7.6  1.3  3.4  0.5   
1   2     6  2012          29  61  13   1.3  64.4  4.1  7.6    1  3.9  0.4   
2   3     6  2012          26  82  22  13.1  47.1  2.5  7.1  0.3  2.7  0.1   

0   Classes  
0  not fire  
1  not fire  
2  not fire  


In [11]:
# Preparando as células
features = ['Temperature', 'RH', 'Ws', 'Rain', 'FFMC', 'DMC', 'DC']

for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=features + ['Classes'])

X = df[features]
y = df['Classes'].map({'fire': 1, 'not fire': 0})

print(f"Amostras: {len(X)}")
print(X.describe())

Amostras: 243
0      Temperature          RH          Ws        Rain        FFMC  \
count   243.000000  243.000000  243.000000  243.000000  243.000000   
mean     32.152263   62.041152   15.493827    0.762963   77.842387   
std       3.628039   14.828160    2.811385    2.003207   14.349641   
min      22.000000   21.000000    6.000000    0.000000   28.600000   
25%      30.000000   52.500000   14.000000    0.000000   71.850000   
50%      32.000000   63.000000   15.000000    0.000000   83.300000   
75%      35.000000   73.500000   17.000000    0.500000   88.300000   
max      42.000000   90.000000   29.000000   16.800000   96.000000   

0             DMC          DC  
count  243.000000  243.000000  
mean    14.680658   49.430864  
std     12.393040   47.665606  
min      0.700000    6.900000  
25%      5.800000   12.350000  
50%     11.300000   33.100000  
75%     20.800000   69.100000  
max     65.900000  220.400000  


In [12]:
# Treinamento do modelo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

print(f"Acurácia: {model.score(X_test, y_test):.2%}")
print("\nRelatório:")
print(classification_report(y_test, model.predict(X_test),
      target_names=['not fire', 'fire']))

Acurácia: 95.92%

Relatório:
              precision    recall  f1-score   support

    not fire       0.95      0.95      0.95        21
        fire       0.96      0.96      0.96        28

    accuracy                           0.96        49
   macro avg       0.96      0.96      0.96        49
weighted avg       0.96      0.96      0.96        49



In [14]:
# Teste de funcionalidade
sample = pd.DataFrame([{
    'Temperature': 38, 'RH': 25, 'Ws': 20,
    'Rain': 0, 'FFMC': 85, 'DMC': 45, 'DC': 150
}])
pred = model.predict(sample)[0]
prob = model.predict_proba(sample)[0]
print(f"Resultado: {'fire' if pred == 1 else 'not fire'}")
print(f"Probabilidade Fire: {prob[1]:.2%}")

Resultado: fire
Probabilidade Fire: 100.00%


In [13]:
# Salva os dados
joblib.dump(model, 'modelo.pkl')
joblib.dump(features, 'features.pkl')
print("Salvos!")

Salvos!
